# K3D Interactive Clipping Planes - Jupyter Solution

This notebook provides **WORKING** clipping plane controls using Jupyter widgets.
No export issues - everything stays in Python!

In [2]:
import numpy as np
import k3d
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets
from IPython.display import display
import time

## Load and Prepare Data

In [3]:
# Load your diffuse intensity data
data = np.load('torch_grid_results.npz')
intensity = data['intensity'].reshape(data['map_shape'])

# Clean data (percentile clipping + NaN handling)
valid = intensity[~np.isnan(intensity)]
vmin, vmax = np.percentile(valid, [1, 99])
intensity = np.clip(intensity, vmin, vmax)
intensity = np.nan_to_num(intensity, nan=vmin)

# Normalize to 0-1
intensity_norm = ((intensity - vmin) / (vmax - vmin)).astype(np.float32)

h, k, l = intensity.shape
print(f"Data shape: {h} × {k} × {l} = {h*k*l:,} voxels")
print(f"Data range: [{intensity.min():.2f}, {intensity.max():.2f}]")
print(f"Normalized to: [0, 1]")

Data shape: 41 × 41 × 41 = 68,921 voxels
Data range: [77.98, 61453.75]
Normalized to: [0, 1]


## Create K3D Plot with Volume

In [8]:
# Create k3d plot
plot = k3d.plot(
    height=600,
    grid_visible=False,
    menu_visibility=True,
    antialias=True
)

# Create volume with optimal settings
volume = k3d.volume(
    intensity_norm,
    color_map=k3d.basic_color_maps.Jet,
    color_range=[0.0, 1.0],
    alpha_coef=30.0,
    bounds=[-h/2, h/2, -k/2, k/2, -l/2, l/2],  # Center at origin
    name="Diffuse Intensity"
)

plot += volume
plot.display()

# Create interactive widgets
def update_clipping(axis='Z', position=50, enable=True):
    """Update clipping plane based on widget values."""
    
    if not enable:
        plot.clipping_planes = []
        return
    
    # Convert position (0-100) to actual coordinate
    bounds = {'X': h, 'Y': k, 'Z': l}
    max_val = bounds[axis] / 2
    distance = -max_val + (max_val * 2 * position / 100)
    
    # Set normal vector based on axis
    normals = {
        'X': [1, 0, 0],
        'Y': [0, 1, 0],
        'Z': [0, 0, 1]
    }
    normal = normals[axis]
    
    # Apply clipping plane
    plot.clipping_planes = [[normal[0], normal[1], normal[2], -distance]]
    
    print(f"Clipping: {axis}-axis at position {position}% (distance={distance:.1f})")

# Create widget controls
interact(update_clipping,
         axis=widgets.RadioButtons(
             options=['X', 'Y', 'Z'],
             value='Z',
             description='Axis:',
             disabled=False
         ),
         position=widgets.IntSlider(
             min=0,
             max=100,
             step=1,
             value=50,
             description='Position:',
             continuous_update=True
         ),
         enable=widgets.Checkbox(
             value=True,
             description='Enable Clipping',
         )
);

Output()

interactive(children=(RadioButtons(description='Axis:', index=2, options=('X', 'Y', 'Z'), value='Z'), IntSlide…

## Interactive Clipping Controls

Use these widgets to control clipping planes in real-time!

interactive(children=(RadioButtons(description='Axis:', index=2, options=('X', 'Y', 'Z'), value='Z'), IntSlide…

## Animated Clipping

Run this cell to see an animation of the clipping plane moving through the volume.

In [6]:
# Animated clipping demonstration
def animate_clipping(axis='Z', speed=0.05, frames=40):
    """Animate clipping plane moving through volume."""
    
    print(f"Animating {axis}-axis clipping...")
    
    bounds = {'X': h, 'Y': k, 'Z': l}
    max_val = bounds[axis] / 2
    
    normals = {
        'X': [1, 0, 0],
        'Y': [0, 1, 0],
        'Z': [0, 0, 1]
    }
    normal = normals[axis]
    
    # Animate forward
    for i in range(frames):
        t = i / (frames - 1)
        distance = -max_val + (2 * max_val * t)
        plot.clipping_planes = [[normal[0], normal[1], normal[2], -distance]]
        time.sleep(speed)
    
    # Animate backward
    for i in range(frames-1, -1, -1):
        t = i / (frames - 1)
        distance = -max_val + (2 * max_val * t)
        plot.clipping_planes = [[normal[0], normal[1], normal[2], -distance]]
        time.sleep(speed)
    
    # Clear clipping
    plot.clipping_planes = []
    print("Animation complete!")

# Create animation controls
animate_button = widgets.Button(
    description='Animate Z-axis',
    button_style='success',
    tooltip='Click to animate clipping plane'
)

def on_animate_click(b):
    animate_clipping('Z', speed=0.03, frames=30)

animate_button.on_click(on_animate_click)
display(animate_button)

Button(button_style='success', description='Animate Z-axis', style=ButtonStyle(), tooltip='Click to animate cl…

## Multiple Clipping Planes

Control multiple clipping planes simultaneously.

In [7]:
# Multiple clipping planes control
def update_multiple_clipping(x_clip=False, x_pos=50, 
                            y_clip=False, y_pos=50,
                            z_clip=False, z_pos=50):
    """Control multiple clipping planes."""
    
    planes = []
    
    if x_clip:
        distance = -h/2 + (h * x_pos / 100)
        planes.append([1, 0, 0, -distance])
    
    if y_clip:
        distance = -k/2 + (k * y_pos / 100)
        planes.append([0, 1, 0, -distance])
    
    if z_clip:
        distance = -l/2 + (l * z_pos / 100)
        planes.append([0, 0, 1, -distance])
    
    plot.clipping_planes = planes
    
    active = []
    if x_clip: active.append(f"X@{x_pos}%")
    if y_clip: active.append(f"Y@{y_pos}%")
    if z_clip: active.append(f"Z@{z_pos}%")
    
    print(f"Active planes: {', '.join(active) if active else 'None'}")

# Create multiple plane controls
interact(update_multiple_clipping,
         x_clip=widgets.Checkbox(value=False, description='X-axis'),
         x_pos=widgets.IntSlider(min=0, max=100, value=50, description='X pos:'),
         y_clip=widgets.Checkbox(value=False, description='Y-axis'),
         y_pos=widgets.IntSlider(min=0, max=100, value=50, description='Y pos:'),
         z_clip=widgets.Checkbox(value=False, description='Z-axis'),
         z_pos=widgets.IntSlider(min=0, max=100, value=50, description='Z pos:')
);

interactive(children=(Checkbox(value=False, description='X-axis'), IntSlider(value=50, description='X pos:'), …

## Preset Clipping Configurations

Quick access to useful clipping configurations.

In [9]:
# Preset configurations
def apply_preset(preset):
    """Apply preset clipping configuration."""
    
    presets = {
        'None': [],
        'X-half': [[1, 0, 0, 0]],
        'Y-half': [[0, 1, 0, 0]],
        'Z-half': [[0, 0, 1, 0]],
        'Corner': [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0]],
        'Diagonal': [[1, 1, 1, 0]],
        'Box': [[1, 0, 0, -h/4], [0, 1, 0, -k/4], [0, 0, 1, -l/4],
                [-1, 0, 0, -h/4], [0, -1, 0, -k/4], [0, 0, -1, -l/4]]
    }
    
    plot.clipping_planes = presets[preset]
    print(f"Applied preset: {preset}")

# Create preset buttons
preset_dropdown = widgets.Dropdown(
    options=['None', 'X-half', 'Y-half', 'Z-half', 'Corner', 'Diagonal', 'Box'],
    value='None',
    description='Preset:',
)

def on_preset_change(change):
    apply_preset(change['new'])

preset_dropdown.observe(on_preset_change, names='value')
display(preset_dropdown)

Dropdown(description='Preset:', options=('None', 'X-half', 'Y-half', 'Z-half', 'Corner', 'Diagonal', 'Box'), v…

## Custom Clipping Plane

Define a custom clipping plane with arbitrary normal vector and distance.

In [11]:
# Create k3d plot
plot = k3d.plot(
    height=600,
    grid_visible=False,
    menu_visibility=True,
    antialias=True
)

# Create volume with optimal settings
volume = k3d.volume(
    intensity_norm,
    color_map=k3d.basic_color_maps.Jet,
    color_range=[0.0, 1.0],
    alpha_coef=30.0,
    bounds=[-h/2, h/2, -k/2, k/2, -l/2, l/2],  # Center at origin
    name="Diffuse Intensity"
)

plot += volume
plot.display()

# Custom clipping plane control
def set_custom_plane(nx=1.0, ny=0.0, nz=0.0, distance=0.0, normalize=True):
    """Set custom clipping plane with arbitrary orientation."""
    
    if normalize:
        # Normalize the normal vector
        norm = np.sqrt(nx**2 + ny**2 + nz**2)
        if norm > 0:
            nx, ny, nz = nx/norm, ny/norm, nz/norm
    
    plot.clipping_planes = [[nx, ny, nz, distance]]
    print(f"Custom plane: normal=({nx:.2f}, {ny:.2f}, {nz:.2f}), distance={distance:.2f}")

# Create custom plane controls
interact(set_custom_plane,
         nx=widgets.FloatSlider(min=-1, max=1, value=1, step=0.1, description='Normal X:'),
         ny=widgets.FloatSlider(min=-1, max=1, value=0, step=0.1, description='Normal Y:'),
         nz=widgets.FloatSlider(min=-1, max=1, value=0, step=0.1, description='Normal Z:'),
         distance=widgets.FloatSlider(min=-50, max=50, value=0, step=1, description='Distance:'),
         normalize=widgets.Checkbox(value=True, description='Normalize')
);

Output()

interactive(children=(FloatSlider(value=1.0, description='Normal X:', max=1.0, min=-1.0), FloatSlider(value=0.…

## Save Current View

Export the current view with clipping planes applied (Note: clipping may not export correctly to HTML).

In [ ]:
# Save current view
def save_view(filename='k3d_view.html'):
    """Save current view to HTML file."""
    
    # Note: As discovered, clipping planes won't export to HTML
    # But this saves the volume and camera position
    
    with open(filename, 'w') as f:
        f.write(plot.get_snapshot())
    
    print(f"View saved to {filename}")
    print("NOTE: Clipping planes are NOT preserved in HTML export (k3d limitation)")
    print("Use this notebook for interactive clipping control")

save_button = widgets.Button(
    description='Save View',
    button_style='primary',
    tooltip='Save current view to HTML'
)

def on_save_click(b):
    save_view('k3d_jupyter_view.html')

save_button.on_click(on_save_click)
display(save_button)

## Direct Python Control

You can also control clipping planes directly in Python:

In [ ]:
# Examples of direct control

# Single clipping plane
plot.clipping_planes = [[1, 0, 0, 0]]  # Clip at X=0
print("Set X-axis clipping at origin")

# Multiple clipping planes
# plot.clipping_planes = [[1, 0, 0, 0], [0, 1, 0, 0]]  # Clip at X=0 and Y=0

# Clear clipping
# plot.clipping_planes = []

## Summary

This Jupyter notebook provides **fully working** interactive clipping plane controls:

✅ **Real-time updates** - Changes apply immediately to the 3D view  
✅ **Multiple control methods** - Sliders, buttons, presets, custom planes  
✅ **Animation support** - Automated clipping plane movement  
✅ **Multi-plane support** - Control multiple clipping planes simultaneously  
✅ **No export issues** - Everything stays in the Jupyter environment  

This is the most reliable way to work with k3d clipping planes!